In [ ]:
# Código: Leitura robusta, limpeza avançada e análise por modelo
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from google.colab import files

# ---------------------------
# Upload (Colab) - descomente se usar
# ---------------------------
# uploaded = files.upload()
# file_name = list(uploaded.keys())[0]

# Ou defina direto:
file_name = "Patriots_2.csv"  # ajuste se necessário

# ---------------------------
# Leitura segura (todos como string)
# ---------------------------
df_raw = pd.read_csv(file_name, sep=';', encoding='latin1', dtype=str, low_memory=False)
print("Linhas no CSV (raw):", len(df_raw))

# ---------------------------
# Função de limpeza numérica robusta
# ---------------------------
def fix_number_v2(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == '':
        return np.nan
    # normalizar espaços (inclui NBSP)
    s = re.sub(r'[\u00A0\s]+', '', s)
    # substituir vírgula por ponto decimal (mas remoção de milhares antes)
    # remover textos como 'km/h', 'rpm', etc.
    s = s.lower()
    # remover unidades e letras
    s = re.sub(r'[a-z%ªº°/\\]', '', s)
    # trocar vírgula por ponto; remover milhares (pontos entre dígitos)
    s = s.replace(',', '.')
    # remover parênteses e traços não-numéricos
    s = s.replace('(', '').replace(')', '')
    # lidar com sinal unicode (menos longo)
    s = s.replace('−', '-')
    # manter apenas dígitos, ponto, e sinal inicial
    m = re.match(r'^-?[\d\.]+', s)
    if not m:
        return np.nan
    token = m.group(0)
    # se houver múltiplos pontos, remover os extras (assumir último ponto é decimal)
    if token.count('.') > 1:
        parts = token.split('.')
        token = ''.join(parts[:-1]) + '.' + parts[-1]
    try:
        return float(token)
    except:
        return np.nan

# ---------------------------
# Padronizar nomes de colunas (corrigir possíveis encodings/BOM)
# ---------------------------
cols_map = {
    '\ufeffData/hora': 'Data/hora',
    'ï»¿Data/hora': 'Data/hora',
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)',
    'Média de consumo (L/h)': 'Média de consumo (L/h)',
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)',
}
df_raw = df_raw.rename(columns={c:v for c,v in cols_map.items() if c in df_raw.columns})

# ---------------------------
# Mostrar colunas e colunas problemáticas
# ---------------------------
print("Colunas detectadas:", df_raw.columns.tolist())

# ---------------------------
# Tratar coluna Modelo: strip, remover BOMs e caracteres invisíveis
# ---------------------------
if 'Modelo' in df_raw.columns:
    df_raw['Modelo_raw'] = df_raw['Modelo'].astype(str)
    df_raw['Modelo'] = df_raw['Modelo_raw'].str.strip().str.replace(r'[\u00A0]', '', regex=True)
    # remover textos, manter dígitos (se houver); manter strings vazias como NaN
    df_raw['Modelo'] = df_raw['Modelo'].replace({'nan': np.nan})
else:
    df_raw['Modelo'] = 'Todos'

# ---------------------------
# Aplicar limpeza numérica nas colunas que interessam
# ---------------------------
numeric_cols = ["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM", "MR", "CL", "UCL", "LCL"]
for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col + '_raw'] = df_raw[col]
        df_raw[col] = df_raw[col].apply(fix_number_v2)
    else:
        print(f"Atenção: coluna '{col}' não encontrada no CSV.")

# ---------------------------
# Converter Data/hora se existir
# ---------------------------
if 'Data/hora' in df_raw.columns:
    df_raw['Data/hora'] = pd.to_datetime(df_raw['Data/hora'], errors='coerce')

# ---------------------------
# Diagnóstico: quantos NaNs por coluna após limpeza
# ---------------------------
print("\nContagem de NaNs por coluna (após limpeza):")
print(df_raw[[c for c in df_raw.columns if c.endswith('_raw') == False]].isna().sum())

# Mostrar amostras de valores problemáticos (exemplos)
for col in ["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM"]:
    if col in df_raw.columns:
        bad = df_raw[df_raw[col].isna()][col + '_raw'] if (col + '_raw') in df_raw.columns else None
        print(f"\nAmostra de valores não convertidos para {col} (até 10):")
        if bad is not None:
            print(bad.dropna().unique()[:10])
        else:
            print("nenhum raw disponível")

# ---------------------------
# Remover linhas sem atributos críticos
# ---------------------------
df = df_raw.dropna(subset=["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM"]).copy()
print("\nLinhas após dropna de campos críticos:", len(df))

# ---------------------------
# Normalização heurística de 'Modelo' (reduzir ruído)
#    - converte valores numéricos em int quando possível
#    - tentativa: se muitos modelos terminam com '0' extra (e divisão por 10 reduz categorias),
#      aplica divisão iterativa por 10 (somente se reduzir o número de categorias)
# ---------------------------
def try_normalize_model_series(s):
    s0 = s.dropna().unique()
    # tentar extrair número inteiro
    def to_digits(x):
        if pd.isna(x): return np.nan
        t = re.sub(r'\D', '', str(x))
        return t if t != '' else np.nan
    num = s.apply(to_digits)
    # se nenhum dígito, retorna original
    if num.dropna().empty:
        return s
    # iterativamente testar divisão por 10 enquanto reduzir categorias
    prev_unique = set(num.dropna())
    series_numeric = num.copy()
    while True:
        # dividir por 10 possíveis valores acabados com '0'
        def div10_if_possible(x):
            if pd.isna(x): return x
            try:
                v = int(x)
            except:
                return x
            if v % 10 == 0:
                return str(v // 10)
            else:
                return str(v)
        new_series = series_numeric.apply(div10_if_possible)
        new_unique = set(new_series.dropna())
        if len(new_unique) < len(prev_unique):
            series_numeric = new_series
            prev_unique = new_unique
        else:
            break
    # se resultou em menos categorias, use; senão, voltar para versão original com dígitos
    if len(prev_unique) < len(set(num.dropna())):
        return series_numeric
    else:
        return num

df['Modelo_norm'] = try_normalize_model_series(df['Modelo'])

# Substituir NaNs de modelo por 'Todos'
df['Modelo_norm'] = df['Modelo_norm'].fillna('Todos')

print("\nContagem por modelos (após normalização heurística):")
print(df['Modelo_norm'].value_counts().head(30))

# ---------------------------
# Agora análise por modelo (control chart + regressão simples)
# ---------------------------
def control_chart(series):
    x = np.array(series)
    if len(x) < 2:
        return None, None, None
    mr = np.abs(np.diff(x))
    mean_x = np.mean(x)
    mean_mr = np.mean(mr)
    # sigma from avg MR: sigma = avg_mr / d2 ; for individuals d2≈1.128
    sigma = mean_mr / 1.128 if mean_mr > 0 else 0
    UCL = mean_x + 3 * sigma
    LCL = max(mean_x - 3 * sigma, 0)
    return mean_x, UCL, LCL

def regressao_simples(df_modelo, modelo):
    X = df_modelo[["Velocidade média (Km/h)", "RPM"]].copy()
    y = df_modelo["Média de consumo (L/h)"].copy()
    reg = LinearRegression()
    reg.fit(X, y)
    return reg

for modelo in df['Modelo_norm'].unique():
    sub = df[df['Modelo_norm'] == modelo]
    print("\n==========================")
    print("MODELO:", modelo, "| Linhas:", len(sub))
    if len(sub) < 10:
        print("Poucos dados — pulando.")
        continue
    mean_x, UCL, LCL = control_chart(sub["Média de consumo (L/h)"])
    print(f"CL: {mean_x:.2f}, UCL: {UCL:.2f}, LCL: {LCL:.2f}")
    reg = regressao_simples(sub, modelo)
    print(f"Regressão: Consumo = {reg.intercept_:.4f} + {reg.coef_[0]:.4f}*Vel + {reg.coef_[1]:.4f}*RPM")


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload do CSV
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name, sep=';', encoding='latin1')

# Renomear colunas para corrigir problemas de codificação
df = df.rename(columns={
    '\ufeffData/hora': 'Data/hora',
    'Velocidade média (Km/h)': 'Velocidade média (Km/h)',
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)',
    'Média de consumo (L/h)': 'Média de consumo (L/h)',
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)',
    'RPM': 'RPM'
})

# Imprimir as colunas para depuração
print("Colunas do DataFrame:")
print(df.columns.tolist())

# Converter 'Data/hora' para datetime se a coluna existir
if 'Data/hora' in df.columns:
    df['Data/hora'] = pd.to_datetime(df['Data/hora'], errors='coerce')
else:
    print("Coluna 'Data/hora' não encontrada. Não convertendo para datetime.")

# Limpar dados numéricos
df['Velocidade média (Km/h)'] = pd.to_numeric(df['Velocidade média (Km/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['Média de consumo (L/h)'] = pd.to_numeric(df['Média de consumo (L/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['RPM'] = pd.to_numeric(df['RPM'].astype(str).str.replace(',', '.'), errors='coerce')

# Remover linhas com valores nulos nas colunas chave
df = df.dropna(subset=['Velocidade média (Km/h)', 'Média de consumo (L/h)', 'RPM'])

# Verificar se coluna 'Modelo' existe; se não, rodar no todo
if 'Modelo' not in df.columns:
    print("Aviso: Coluna 'Modelo' não encontrada. Rodando análise no dataset completo.")
    df['Modelo'] = 'Todos'

# Função para calcular os valores da carta de controle por modelo
def calculate_control_chart(sub_df, model_name):
    if len(sub_df) < 2:  # Precisa de pelo menos 2 pontos para faixa móvel
        print(f"\nModelo {model_name}: Poucos dados ({len(sub_df)} linhas). Pulando análise.")
        return

    # Ordenar por data/hora se a coluna existir
    if 'Data/hora' in sub_df.columns:
        sub_df = sub_df.sort_values('Data/hora').reset_index(drop=True)
    else:
        print(f"Aviso para {model_name}: Sem coluna 'Data/hora', não ordenando por tempo.")

    # CL: Média de consumo
    cl = sub_df['Média de consumo (L/h)'].mean()

    # Faixa Móvel (Moving Range): Diferenças absolutas consecutivas
    sub_df['MR'] = sub_df['Média de consumo (L/h)'].diff().abs()
    avg_mr = sub_df['MR'].mean()  # Ignora o primeiro NaN automaticamente

    # Desvio Padrão Estimado: Avg MR / 1.128 (constante para individuals chart)
    sigma = avg_mr / 1.128

    # UCL: CL + 3 * sigma
    ucl = cl + 3 * sigma

    # LCL: CL - 3 * sigma (máximo 0 para consumo)
    lcl = max(cl - 3 * sigma, 0)

    # Exibir resultados
    print(f'\nResultados para {model_name}:')
    print(f'CL (Média de consumo): {cl:.2f} L/h')
    print(f'Média faixa móvel: {avg_mr:.2f}')
    print(f'Desvio padrão estimado: {sigma:.2f}')
    print(f'Limite superior de controle (UCL): {ucl:.2f}')
    print(f'Limite inferior de controle (LCL): {lcl:.2f}')

# Rodar para cada modelo
models = df['Modelo'].unique()
for m in models:
    sub_df = df[df['Modelo'] == m]
    calculate_control_chart(sub_df, m)

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Upload do CSV (agora com coluna 'Modelo' opcional)
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name, sep=';', encoding='latin1') # Added encoding='latin1' to handle potential character issues

# Renomear colunas para corrigir problemas de codificação
df = df.rename(columns={
    '\ufeffData/hora': 'Data/hora', # Handle BOM if present
    'Velocidade média (Km/h)': 'Velocidade média (Km/h)', # Ensure exact match
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)', # Handle common misencoding of 'é'
    'Média de consumo (L/h)': 'Média de consumo (L/h)', # Ensure exact match
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)', # Handle common misencoding of 'é'
    'RPM': 'RPM' # Ensure exact match
})

# Limpar dados: Usar pd.to_numeric com errors='coerce' para lidar com valores não numéricos
df['Velocidade média (Km/h)'] = pd.to_numeric(df['Velocidade média (Km/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['Média de consumo (L/h)'] = pd.to_numeric(df['Média de consumo (L/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['RPM'] = pd.to_numeric(df['RPM'].astype(str).str.replace(',', '.'), errors='coerce')

# Remover linhas com valores nulos nas colunas chave após a conversão
df = df.dropna(subset=['Velocidade média (Km/h)', 'Média de consumo (L/h)', 'RPM'])

# Verificar se coluna 'Modelo' existe; se não, rodar no todo
if 'Modelo' not in df.columns:
    print("Aviso: Coluna 'Modelo' não encontrada. Rodando análise no dataset completo.")
    models = ['Todos']
    df['Modelo'] = 'Todos'  # Adiciona temporariamente
else:
    models = df['Modelo'].unique()
    print(f"Modelos encontrados: {models}")

# Função para análise por modelo
def analyze_model(sub_df, model_name):
    if len(sub_df) < 10:
        print(f"\nModelo {model_name}: Poucos dados ({len(sub_df)} linhas). Pulando análise.")
        return None, None, None

    # Correlação
    corr_matrix = sub_df[['Velocidade média (Km/h)', 'Média de consumo (L/h)', 'RPM']].corr()
    print(f'\nMatriz de Correlação para {model_name}:')
    print(corr_matrix)

    # Regressão
    X = sub_df[['Velocidade média (Km/h)', 'RPM']]
    X = sm.add_constant(X)
    y = sub_df['Média de consumo (L/h)']
    reg_model = sm.OLS(y, X).fit()
    print(f'\nResumo da Regressão para {model_name}:')
    print(reg_model.summary())

    # Exibir a equação do modelo explicitamente
    print(f'\nEquação do Modelo para {model_name}:')
    print(f'Consumo (L/h) ≈ {reg_model.params["const"]:.2f} + {reg_model.params["Velocidade média (Km/h)"]:.2f} × Velocidade (km/h) + {reg_model.params["RPM"]:.2f} × RPM')

    # Otimização (grid search)
    vel_range = np.linspace(sub_df['Velocidade média (Km/h)'].min(), sub_df['Velocidade média (Km/h)'].max(), 50)
    rpm_range = np.linspace(sub_df['RPM'].min(), sub_df['RPM'].max(), 50)
    vel_grid, rpm_grid = np.meshgrid(vel_range, rpm_range)
    consumo_grid = reg_model.params['const'] + reg_model.params['Velocidade média (Km/h)'] * vel_grid + reg_model.params['RPM'] * rpm_grid

    min_idx = np.unravel_index(np.argmin(consumo_grid), consumo_grid.shape)
    min_vel = vel_range[min_idx[1]]
    min_rpm = rpm_range[min_idx[0]]
    min_consumo = consumo_grid[min_idx]

    print(f'\nCombinação Ótima para {model_name}:')
    print(f'Velocidade: {min_vel:.2f} km/h')
    print(f'RPM: {min_rpm:.2f}')
    print(f'Consumo Estimado: {min_consumo:.2f} L/h')

    # Tabela de sugestões
    optim_df = pd.DataFrame({
        'Velocidade (km/h)': [min_vel, min_vel + 1.5, min_vel + 3.5, min_vel + 6.5],
        'RPM': [min_rpm, min_rpm - 10, min_rpm - 20, min_rpm - 30],
    })
    optim_df['Consumo Estimado (L/h)'] = reg_model.params['const'] + reg_model.params['Velocidade média (Km/h)'] * optim_df['Velocidade (km/h)'] + reg_model.params['RPM'] * optim_df['RPM']
    print(f'\nSugestões para {model_name}:')
    print(optim_df)

    # Gráficos
    plt.figure(figsize=(12, 4))
    plt.suptitle(f'Análise para {model_name}')

    plt.subplot(1, 3, 1)
    sns.scatterplot(x='Velocidade média (Km/h)', y='Média de consumo (L/h)', data=sub_df)
    sns.regplot(x='Velocidade média (Km/h)', y='Média de consumo (L/h)', data=sub_df, scatter=False, color='red')
    plt.title('Velocidade vs. Consumo')

    plt.subplot(1, 3, 2)
    sns.scatterplot(x='RPM', y='Média de consumo (L/h)', data=sub_df)
    sns.regplot(x='RPM', y='Média de consumo (L/h)', data=sub_df, scatter=False, color='red')
    plt.title('RPM vs. Consumo')

    plt.subplot(1, 3, 3)
    sns.scatterplot(x='Velocidade média (Km/h)', y='RPM', data=sub_df)
    plt.title('Velocidade vs. RPM')

    plt.tight_layout()
    plt.show()

    return reg_model, min_vel, min_consumo

# Rodar para cada modelo
for m in models:
    sub_df = df[df['Modelo'] == m]
    analyze_model(sub_df, m)

In [ ]:
# ============================================
# 1. IMPORTS
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (10, 6)
sns.set(style="whitegrid")

# ============================================
# 2. LER CSV
# ============================================
file = "/content/Patriots_2.csv"
df = pd.read_csv(file, sep=";", encoding="latin1", low_memory=False)

print("Linhas:", len(df))
print("Colunas:", df.columns.tolist())


# ============================================
# 3. REMOVER COLUNAS LIXO
# ============================================
df = df.loc[:, ~df.columns.str.contains("Unnamed")]


# ============================================
# 4. AJUSTAR NOMES DE COLUNAS COM ERROS DE CODIFICAÇÃO
# ============================================
df = df.rename(columns={
    "Velocidade mÃ©dia (Km/h)": "Velocidade média (Km/h)",
    "MÃ©dia de consumo (L/h)": "Média de consumo (L/h)"
})


# ============================================
# 5. TRATAMENTO DE VALORES INVALIDOS
# ============================================
def limpar_valores(x):
    """Remove texto, símbolos e deixa só números e ponto"""
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()

    # valores claramente inválidos
    if x in ["true", "false", "sim", "não", "na", "nan", "none"]:
        return np.nan

    # troca vírgula por ponto
    x = x.replace(",", ".")

    # remove tudo que não for número, ponto ou sinal
    x = ''.join(c for c in x if c.isdigit() or c in ".-")

    return x if x != "" else np.nan


numeric_cols = [
    "Velocidade média (Km/h)",
    "Média de consumo (L/h)",
    "RPM", "CL", "UCL", "LCL"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].apply(limpar_valores).astype(float)


# ============================================
# 6. NORMALIZAR MODELO
#    Corrigido: não usa "\d" sem escape correto, usa r"\d+"
# ============================================
df["Modelo"] = df["Modelo"].astype(str).str.extract(r"(\d+)").astype(float)


# ============================================
# 7. REMOVER LINHAS COM NAN
# ============================================
df = df.dropna(subset=["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM", "Modelo"])

print("\nLinhas após limpeza:", len(df))


# ============================================
# 8. FUNÇÃO DE REGRESSÃO
# ============================================
def analisar_modelo(df, modelo):

    print("\n\n====================================")
    print(f"          MODELO {modelo}")
    print("====================================")

    data = df[df["Modelo"] == modelo]

    X = data[["Velocidade média (Km/h)", "RPM"]]
    y = data["Média de consumo (L/h)"]

    Xc = sm.add_constant(X)
    model = sm.OLS(y, Xc).fit()

    print(model.summary())

    coef = model.params

    print("\nEquação:")
    print(f"Consumo = {coef['const']:.4f} + "
          f"{coef['Velocidade média (Km/h)']:.4f}*Vel + "
          f"{coef['RPM']:.4f}*RPM")

    # ------- Gráfico: dispersão + regressão -------
    plt.figure(figsize=(10, 5))
    sns.scatterplot(x=X["Velocidade média (Km/h)"], y=y, alpha=0.3)

    vel_range = np.linspace(X["Velocidade média (Km/h)"].min(),
                            X["Velocidade média (Km/h)"].max(), 200)
    rpm_mean = X["RPM"].mean()

    y_pred = (coef["const"]
              + coef["Velocidade média (Km/h)"] * vel_range
              + coef["RPM"] * rpm_mean)

    plt.plot(vel_range, y_pred, linewidth=3)
    plt.title(f"Modelo {modelo} — Regressão")
    plt.xlabel("Velocidade média (Km/h)")
    plt.ylabel("Consumo (L/h)")
    plt.show()

    # ------- Resíduos -------
    residuals = model.resid
    pred = model.predict(Xc)

    plt.figure(figsize=(10, 5))
    plt.scatter(pred, residuals, alpha=0.3)
    plt.axhline(0, color="red")
    plt.title(f"Resíduos — Modelo {modelo}")
    plt.xlabel("Predito")
    plt.ylabel("Resíduo")
    plt.show()

    # ------- Histograma -------
    plt.figure(figsize=(10, 5))
    plt.hist(residuals, bins=60)
    plt.title(f"Histograma de Resíduos — Modelo {modelo}")
    plt.show()

    # ------- Outliers (Z-score) -------
    z = np.abs((residuals - residuals.mean()) / residuals.std())
    outliers = data[z > 3]

    print(f"Outliers detectados (Z > 3): {len(outliers)}")

    if len(outliers) > 0:
        plt.figure(figsize=(10, 5))
        plt.scatter(pred, residuals, alpha=0.3)
        plt.scatter(pred[z > 3], residuals[z > 3], color="red", label="Outliers")
        plt.legend()
        plt.title(f"Outliers — Modelo {modelo}")
        plt.show()


# ============================================
# 9. RODAR POR MODELO
# ============================================
for m in sorted(df["Modelo"].unique()):
    analisar_modelo(df, m)


In [ ]:
# ============================
# Gráficos, R², resíduos e outliers por modelo
# Cole este bloco no Colab e execute
# ============================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# Configs visuais
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# ---------- AJUSTE: caminho do arquivo ----------
FILEPATH = "/content/Patriots_2.csv"   # ajuste se necessário

# ---------- LEITURA ----------
df = pd.read_csv(FILEPATH, sep=';', encoding='latin1', low_memory=False)
print("Linhas:", len(df))
print("Colunas:", df.columns.tolist())

# ---------- LIMPEZA BÁSICA ----------
# remover colunas 'Unnamed'
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# normalizar possíveis variações de encoding (mas você confirmou A)
col_map = {
    "Velocidade mÃ©dia (Km/h)": "Velocidade média (Km/h)",
    "MÃ©dia de consumo (L/h)": "Média de consumo (L/h)"
}
df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

# Função robusta para limpar valores numéricos (retorna np.nan quando inválido)
def limpar_valor_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["", "nan", "na", "none", "false", "true", "sim", "nao", "não"]:
        return np.nan
    # substituir vírgula por ponto
    s = s.replace(",", ".")
    # remover unidades e letras
    s = "".join(ch for ch in s if (ch.isdigit() or ch in ".- ")) # Keep space for potential future cleaning
    s = s.replace(" ", "") # Remove spaces after filtering
    if s == "" or s in ["-", ".", "-."]:
        return np.nan
    # lidar com múltiplos pontos
    if s.count('.') > 1:
        parts = s.split('.')
        s = ''.join(parts[:-1]) + '.' + parts[-1]
    try:
        return float(s)
    except:
        return np.nan

# Converter colunas críticas
critical_cols = ["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM"]
for c in critical_cols:
    if c in df.columns:
        df[c] = df[c].apply(limpar_valor_num)
    else:
        raise ValueError(f"Coluna crítica '{c}' não encontrada no CSV. Verifique nomes.")

# Normalizar/Extrair modelo numérico (caso contenha texto)
df['Modelo'] = df['Modelo'].astype(str).str.extract(r'(\d+)')
df['Modelo'] = df['Modelo'].astype(float)  # permite NaN se não extraiu número

# remover linhas sem os campos essenciais
df = df.dropna(subset=critical_cols + ['Modelo']).copy()

# converter Modelo para int (se quiser) — mas mantemos float como chave
df['Modelo'] = df['Modelo'].astype(int)

print("Linhas após limpeza:", len(df))
print("Modelos encontrados:", sorted(df["Modelo"].unique()))

# pasta para salvar figuras
OUTDIR_ROOT = "/content/figs_by_model"
os.makedirs(OUTDIR_ROOT, exist_ok=True)

# ---------- Função de análise por modelo ----------
def analizar_y_plot(df_model, modelo, outdir):
    os.makedirs(outdir, exist_ok=True)

    X = df_model[["Velocidade média (Km/h)", "RPM"]].copy()
    y = df_model["Média de consumo (L/h)"].copy()

    # Ajuste OLS (com intercepto)
    Xc = sm.add_constant(X)
    model = sm.OLS(y, Xc).fit()

    # Estatísticas chave
    r2 = model.rsquared
    params = model.params
    n_obs = len(df_model)
    print("\n" + "="*60)
    print(f"MODELO {modelo} — Observações: {n_obs}")
    print(f"R²: {r2:.4f}")
    print("Equação:")
    print(f"Consumo = {params['const']:.4f} + {params['Velocidade média (Km/h)']:.6f}*Vel + {params['RPM']:.6f}*RPM")
    print(model.summary())

    # Predições e resíduos
    preds = model.predict(Xc)
    residuals = model.resid

    # Detectar outliers por Z-score do resíduo
    # Convert z_res to a Series with the same index as residuals for easy boolean indexing
    z_res_series = pd.Series(np.abs((residuals - residuals.mean()) / residuals.std(ddof=0)), index=residuals.index)
    is_outlier_z = z_res_series > 3
    n_outliers_z = is_outlier_z.sum()

    # Detectar outliers por IQR no resíduo
    q1, q3 = np.percentile(residuals, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    is_outlier_iqr = (residuals < lower) | (residuals > upper)
    n_outliers_iqr = is_outlier_iqr.sum()

    print(f"Outliers (Z-score > 3): {n_outliers_z}")
    print(f"Outliers (IQR): {n_outliers_iqr}")

    # --- 1) Scatter Velocidade vs Consumo com linha de regressão (RPM = mediana) ---
    plt.figure(figsize=(10,6))
    sns.scatterplot(x=df_model["Velocidade média (Km/h)"], y=y, alpha=0.18, s=8)
    vel_range = np.linspace(df_model["Velocidade média (Km/h)"].min(), df_model["Velocidade média (Km/h)"].max(), 300)
    rpm_med = df_model["RPM"].median()
    y_line = params['const'] + params['Velocidade média (Km/h)'] * vel_range + params['RPM'] * rpm_med
    plt.plot(vel_range, y_line, color='red', linewidth=2.5, label=f"Regressão (RPM med={rpm_med:.0f})")
    plt.title(f"Modelo {modelo} — Velocidade vs Consumo (n={n_obs})")
    plt.xlabel("Velocidade média (Km/h)")
    plt.ylabel("Média de consumo (L/h)")
    plt.legend()
    plt.tight_layout()
    fn = os.path.join(outdir, f"scatter_regressao_modelo_{modelo}.png")
    plt.savefig(fn, dpi=150)
    plt.show()

    # --- 2) Resíduos vs Preditos ---
    plt.figure(figsize=(10,6))
    plt.scatter(preds, residuals, alpha=0.18, s=8)
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f"Modelo {modelo} — Resíduos vs Preditos")
    plt.xlabel("Predito (L/h)")
    plt.ylabel("Resíduo (L/h)")
    fn = os.path.join(outdir, f"residuos_vs_preditos_modelo_{modelo}.png")
    plt.tight_layout()
    plt.savefig(fn, dpi=150)
    plt.show()

    # --- 3) Histograma dos resíduos ---
    plt.figure(figsize=(10,6))
    sns.histplot(residuals, bins=80, kde=True)
    plt.title(f"Modelo {modelo} — Histograma dos Resíduos")
    plt.xlabel("Resíduo (L/h)")
    fn = os.path.join(outdir, f"hist_residuos_modelo_{modelo}.png")
    plt.tight_layout()
    plt.savefig(fn, dpi=150)
    plt.show()

    # --- 4) Boxplot do consumo (com outliers destacados) ---
    plt.figure(figsize=(8,5))
    sns.boxplot(x=y)
    plt.title(f"Modelo {modelo} — Boxplot Consumo (L/h)")
    fn = os.path.join(outdir, f"boxplot_consumo_modelo_{modelo}.png")
    plt.tight_layout()
    plt.savefig(fn, dpi=150)
    plt.show()

    # --- 5) Scatter destacando outliers (Z-score) ---
    plt.figure(figsize=(10,6))
    plt.scatter(df_model["Velocidade média (Km/h)"], y, alpha=0.12, s=8, label='dados')
    if n_outliers_z > 0:
        plt.scatter(df_model[is_outlier_z]["Velocidade média (Km/h)"],
                    y[is_outlier_z],
                    color='red', s=15, label=f'Outliers Z>3 ({n_outliers_z})')
    plt.title(f"Modelo {modelo} — Outliers (Z-score)")
    plt.xlabel("Velocidade média (Km/h)")
    plt.ylabel("Consumo (L/h)")
    plt.legend()
    fn = os.path.join(outdir, f"outliers_scatter_modelo_{modelo}.png")
    plt.tight_layout()
    plt.savefig(fn, dpi=150)
    plt.show()

    # --- 6) Salvar tabela de outliers (Z-score) ---
    if n_outliers_z > 0:
        df_outliers_z = df_model[is_outlier_z].copy()
        df_outliers_z['residual'] = residuals[is_outlier_z]
        df_outliers_z['z_resid'] = z_res_series[is_outlier_z]
        csv_out = os.path.join(outdir, f"outliers_z_modelo_{modelo}.csv")
        df_outliers_z.to_csv(csv_out, index=False)
        print("Outliers (Z) salvos em:", csv_out)

    # retornar resumo
    summary = {
        "modelo": modelo,
        "n_obs": n_obs,
        "r2": r2,
        "n_outliers_z": n_outliers_z,
        "n_outliers_iqr": n_outliers_iqr,
        "outdir": outdir
    }
    return summary

# ---------- Rodar para cada modelo (ordenado) ----------
summaries = []
for modelo in sorted(df["Modelo"].unique()):
    outdir = os.path.join(OUTDIR_ROOT, f"modelo_{modelo}")
    df_model = df[df["Modelo"] == modelo].copy()
    summary = analizar_y_plot(df_model, modelo, outdir)
    summaries.append(summary)

# ---------- Resumo final ----------
print("\n\nRESUMO GERAL:")
for s in summaries:
    print(f"Modelo {s['modelo']}: n={s['n_obs']}, R2={s['r2']:.4f}, outliers_z={s['n_outliers_z']}, outliers_iqr={s['n_outliers_iqr']}, figs_dir={s['outdir']}")

print("\nFiguras e CSVs de outliers (se houver) salvos em:", OUTDIR_ROOT)


In [ ]:
# =============================================
# MODELOS NÃO LINEARES PARA CADA MODELO DE MÁQUINA
# Random Forest, Gradient Boosting, XGBoost
# =============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =============================================================
# CARREGAR OS DADOS
# =============================================================
df = pd.read_csv("Patriots_2.csv", sep=';', encoding='latin1')

# Renomear colunas para corrigir problemas de codificação
df = df.rename(columns={
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)',
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)'
})

# Remover colunas vazias
df = df.dropna(axis=1, how="all")

# Remover linhas sem as variáveis necessárias
df = df.dropna(subset=["Velocidade média (Km/h)", "RPM", "Média de consumo (L/h)", "Modelo"])

# Garantir valores numéricos
df["Velocidade média (Km/h)"] = pd.to_numeric(df["Velocidade média (Km/h)"], errors="coerce")
df["RPM"] = pd.to_numeric(df["RPM"], errors="coerce")
df["Média de consumo (L/h)"] = pd.to_numeric(df["Média de consumo (L/h)"], errors="coerce")

df = df.dropna(subset=["Velocidade média (Km/h)", "RPM", "Média de consumo (L/h)"])

print("Modelos encontrados:", df["Modelo"].unique())

# =============================================================
# FUNÇÃO PARA TREINAR MODELOS NÃO LINEARES
# =============================================================
def treinar_modelos(df_modelo, modelo_nome):

    X = df_modelo[["Velocidade média (Km/h)", "RPM"]]
    y = df_modelo["Média de consumo (L/h)"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    resultados = {}

    # ---------------------------
    # Random Forest
    # ---------------------------
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)

    resultados["Random Forest"] = {
        "modelo": rf,
        "R2": r2_score(y_test, y_pred_rf),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        "MAE": mean_absolute_error(y_test, y_pred_rf)
    }

    # ---------------------------
    # Gradient Boosting
    # ---------------------------
    gb = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4
    )
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)

    resultados["Gradient Boosting"] = {
        "modelo": gb,
        "R2": r2_score(y_test, y_pred_gb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        "MAE": mean_absolute_error(y_test, y_pred_gb)
    }

    # ---------------------------
    # XGBoost
    # ---------------------------
    xg = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method="hist"
    )
    xg.fit(X_train, y_train)
    y_pred_xg = xg.predict(X_test)

    resultados["XGBoost"] = {
        "modelo": xg,
        "R2": r2_score(y_test, y_pred_xg),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_xg)),
        "MAE": mean_absolute_error(y_test, y_pred_xg)
    }

    # =============================================================
    # SALVAR IMPORTÂNCIA DAS FEATURES
    # =============================================================
    fig_dir = f"/content/figs_models_naolineares/modelo_{modelo_nome}"
    os.makedirs(fig_dir, exist_ok=True)

    for nome, res in resultados.items():
        try:
            importances = res["modelo"].feature_importances_
            plt.figure(figsize=(6,4))
            sns.barplot(x=importances, y=X.columns)
            plt.title(f"Importância das variáveis — {nome} (Modelo {modelo_nome})")
            plt.tight_layout()
            plt.savefig(f"{fig_dir}/importancia_{nome.replace(' ','_')}.png", dpi=200)
            plt.close()
        except:
            pass

    return resultados

# =============================================================
# RODAR PARA CADA MODELO
# =============================================================
modelos = df["Modelo"].unique()

resultado_final = {}

for m in modelos:
    print("\n=============================================")
    print(f" TREINANDO MODELOS NÃO LINEARES PARA MODELO {m} ")
    print("=============================================\n")

    df_modelo = df[df["Modelo"] == m].copy()
    res = treinar_modelos(df_modelo, m)

    resultado_final[m] = res

# =============================================================
# EXIBIR O RESUMO
# =============================================================
print("\n\n====================== RESULTADOS FINAIS ======================\n")
for m, res in resultado_final.items():
    print(f"\nModelo {m}:")
    for nome, r in res.items():
        print(f"  - {nome}: R²={r['R2']:.4f} | RMSE={r['RMSE']:.2f} | MAE={r['MAE']:.2f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def diagnostico_basico(df, modelo):
    dfm = df[df["Modelo"] == modelo]

    plt.figure(figsize=(16,4))

    plt.subplot(1,3,1)
    sns.histplot(dfm["Velocidade média (Km/h)"], bins=50, kde=True)
    plt.title(f"Distribuição da Velocidade — Modelo {modelo}")

    plt.subplot(1,3,2)
    sns.histplot(dfm["RPM"], bins=50, kde=True)
    plt.title(f"Distribuição do RPM — Modelo {modelo}")

    plt.subplot(1,3,3)
    sns.histplot(dfm["Média de consumo (L/h)"], bins=50, kde=True)
    plt.title(f"Distribuição do Consumo — Modelo {modelo}")

    plt.tight_layout()
    plt.show()

diagnostico_basico(df, 250)
diagnostico_basico(df, 350)


In [ ]:
# =============================================
# MODELOS NÃO LINEARES PARA CADA MODELO DE MÁQUINA
# Random Forest, Gradient Boosting, XGBoost
# =============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =============================================================
# CARREGAR OS DADOS
# =============================================================
df = pd.read_csv("Patriots_2.csv", sep=';', encoding='latin1', low_memory=False)

# Renomear colunas para corrigir problemas de codificação
df = df.rename(columns={
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)',
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)'
})

# Garantir valores numéricos para colunas críticas
df["Velocidade média (Km/h)"] = pd.to_numeric(df["Velocidade média (Km/h)"], errors="coerce")
df["RPM"] = pd.to_numeric(df["RPM"], errors="coerce")
df["Média de consumo (L/h)"] = pd.to_numeric(df["Média de consumo (L/h)"], errors="coerce")

# Remover colunas completamente vazias (após as renomeações e conversões iniciais)
df = df.dropna(axis=1, how="all")

# Remover linhas onde consumo é zero ou nulo
if "Média de consumo (L/h)" in df.columns:
    df = df[df["Média de consumo (L/h)"] > 0]

# Remover linhas com valores nulos nas variáveis necessárias (após a limpeza acima)
df = df.dropna(subset=["Velocidade média (Km/h)", "RPM", "Média de consumo (L/h)", "Modelo"])

# Reset index
df = df.reset_index(drop=True)

print("Modelos encontrados:", df["Modelo"].unique())


# =============================================================
# FUNÇÃO PARA TREINAR MODELOS NÃO LINEARES (SEM CLUSTERING)
# =============================================================
def treinar_modelos_por_modelo(df_modelo, modelo_nome):

    print("\n=============================================")
    print(f" TREINANDO MODELOS N\u00c3O LINEARES PARA MODELO {modelo_nome} ")
    print("=============================================\n")

    # Verificar se há dados suficientes para treinar
    if len(df_modelo) < 10: # Mínimo de 10 linhas para treino/teste
        print(f"Aviso: Poucos dados para o Modelo {modelo_nome} ({len(df_modelo)} linhas). Pulando o treinamento.")
        return {}

    X = df_modelo[["Velocidade média (Km/h)", "RPM"]]
    y = df_modelo["Média de consumo (L/h)"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    resultados = {}

    # ---------------------------
    # Random Forest
    # ---------------------------
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)

    resultados["Random Forest"] = {
        "modelo": rf,
        "R2": r2_score(y_test, y_pred_rf),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        "MAE": mean_absolute_error(y_test, y_pred_rf)
    }

    # ---------------------------
    # Gradient Boosting
    # ---------------------------
    gb = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4
    )
    gb.fit(X_train, y_train)
    y_pred_gb = gb.predict(X_test)

    resultados["Gradient Boosting"] = {
        "modelo": gb,
        "R2": r2_score(y_test, y_pred_gb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        "MAE": mean_absolute_error(y_test, y_pred_gb)
    }

    # ---------------------------
    # XGBoost
    # ---------------------------
    xg = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method="hist"
    )
    xg.fit(X_train, y_train)
    y_pred_xg = xg.predict(X_test)

    resultados["XGBoost"] = {
        "modelo": xg,
        "R2": r2_score(y_test, y_pred_xg),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_xg)),
        "MAE": mean_absolute_error(y_test, y_pred_xg)
    }

    # =============================================================
    # SALVAR IMPORTÂNCIA DAS FEATURES
    # =============================================================
    fig_dir = f"/content/figs_models_naolineares/modelo_{modelo_nome}"
    os.makedirs(fig_dir, exist_ok=True)

    for nome, res in resultados.items():
        try:
            importances = res["modelo"].feature_importances_
            plt.figure(figsize=(6,4))
            sns.barplot(x=importances, y=X.columns)
            plt.title(f"Importância das variáveis — {nome} (Modelo {modelo_nome})")
            plt.tight_layout()
            plt.savefig(f"{fig_dir}/importancia_{nome.replace(' ','_')}.png", dpi=200)
            plt.close() # Fechar a figura para liberar memória
        except Exception as e:
            print(f"Aviso: Não foi possível gerar importância das features para {nome} (Modelo {modelo_nome}): {e}")

    return resultados

# =============================================================
# RODAR PARA CADA MODELO (SEM CLUSTERING)
# =============================================================
modelos = df["Modelo"].unique()

resultado_final = {}

# Filtra modelos NaN da lista para evitar processá-los
modelos_validos = [m for m in modelos if pd.notna(m)]

for m in modelos_validos:
    df_mod = df[df["Modelo"] == m].copy()
    res = treinar_modelos_por_modelo(df_mod, m)
    if res: # Adicionar apenas se houver resultados (se não foi pulado por poucos dados)
        resultado_final[m] = res

# =============================================================
# EXIBIR O RESUMO
# =============================================================
print("\n\n====================== RESULTADOS FINAIS ======================\n")
if not resultado_final:
    print("Nenhum modelo foi treinado devido a dados insuficientes para todos os modelos.\n")
else:
    for m, res in resultado_final.items():
        print(f"\nModelo {m}:")
        for nome, r in res.items():
            print(f"  - {nome}: R²={r['R2']:.4f} | RMSE={r['RMSE']:.2f} | MAE={r['MAE']:.2f}")


In [ ]:
df_raw = pd.read_csv("Patriots_2.csv", sep=";", encoding="latin1", low_memory=False)

print(df_raw.columns)

print("\nVelocidade — primeiros 20 valores:")
print(df_raw.iloc[:20, df_raw.columns.str.contains("Velocidade", case=False)].to_string())

print("\nConsumo — primeiros 20 valores:")
print(df_raw.iloc[:20, df_raw.columns.str.contains("consumo", case=False)].to_string())

print("\nRPM — primeiros 20 valores:")
print(df_raw.iloc[:20, df_raw.columns.str.contains("RPM", case=False)].to_string())


In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("Patriots_2.csv", sep=';', encoding='latin1', low_memory=False)

# --- corrigir nomes das colunas
df.columns = df.columns.str.replace("Ã©", "é").str.replace("Ãº", "ú").str.replace("Ã", "í")
df.columns = df.columns.str.replace("MÃ©dia", "Média")
df.columns = df.columns.str.replace("Velocidade média \(Km/h\)", "Velocidade média (Km/h)")

# --- função robusta para conversão
def fix_num(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace(".", "")      # remove separador de milhar
    x = x.replace(",", ".")     # vira decimal
    x = re.sub(r"[^0-9\.\-]", "", x)  # remove lixo invisível
    try:
        return float(x)
    except:
        return np.nan

cols = ["Velocidade média (Km/h)", "Média de consumo (L/h)", "RPM"]

for c in cols:
    df[c] = df[c].apply(fix_num)

print("Após conversão, NaNs por coluna:")
print(df[cols].isna().sum())
print("\nPrimeiras linhas convertidas:")
print(df[cols].head(10))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import os

# ================================
#  UPLOAD DO ARQUIVO
# ================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name, sep=';', encoding='latin1')

# ================================
#  CORREÇÃO DE COLUNAS
# ================================
df = df.rename(columns={
    '\ufeffData/hora': 'Data/hora',
    'Velocidade média (Km/h)': 'Velocidade média (Km/h)',
    'Velocidade mÃ©dia (Km/h)': 'Velocidade média (Km/h)',
    'Média de consumo (L/h)': 'Média de consumo (L/h)',
    'MÃ©dia de consumo (L/h)': 'Média de consumo (L/h)',
    'RPM': 'RPM'
})

print(df.columns)

# ================================
#  CONVERSÕES
# ================================
if 'Data/hora' in df.columns:
    df['Data/hora'] = pd.to_datetime(df['Data/hora'], errors='coerce')

df['Velocidade média (Km/h)'] = pd.to_numeric(df['Velocidade média (Km/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['Média de consumo (L/h)'] = pd.to_numeric(df['Média de consumo (L/h)'].astype(str).str.replace(',', '.'), errors='coerce')
df['RPM'] = pd.to_numeric(df['RPM'].astype(str).str.replace(',', '.'), errors='coerce')

df = df.dropna(subset=['Média de consumo (L/h)'])

if 'Modelo' not in df.columns:
    df['Modelo'] = 'Todos'

# ================================
#  FUNÇÃO PARA PLOTAR CARTA DE CONTROLE
# ================================
def plot_control_chart(sub_df, model_name):

    # Ordenar temporalmente, se possível
    if 'Data/hora' in sub_df.columns:
        sub_df = sub_df.sort_values('Data/hora')

    consumo = sub_df['Média de consumo (L/h)'].reset_index(drop=True)

    # ================================
    #  CÁLCULO DA CARTA X (INDIVIDUAL)
    # ================================
    CL = consumo.mean()

    MR = consumo.diff().abs()
    MR_mean = MR.mean()

    sigma = MR_mean / 1.128

    UCL = CL + 3 * sigma
    LCL = max(CL - 3 * sigma, 0)

    # ================================
    #  PLOTAR CARTA X
    # ================================
    save_dir = f"/content/control_charts/modelo_{model_name}"
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(16,5))
    plt.plot(consumo, marker='.', linestyle='-', alpha=0.7, label="Consumo")
    plt.axhline(CL, color='blue', linestyle='--', label=f'CL ({CL:.2f})')
    plt.axhline(UCL, color='red', linestyle='--', label=f'UCL ({UCL:.2f})')
    plt.axhline(LCL, color='red', linestyle='--', label=f'LCL ({LCL:.2f})')

    plt.title(f"Carta de Controle — Consumo (Modelo {model_name})")
    plt.xlabel("Observação")
    plt.ylabel("Consumo (L/h)")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.savefig(f"{save_dir}/carta_X_modelo_{model_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # ================================
    #  PLOTAR CARTA MR
    # ================================
    plt.figure(figsize=(16,5))
    plt.plot(MR, marker='.', linestyle='-', alpha=0.7, label="MR")
    plt.axhline(MR_mean, color='blue', linestyle='--', label=f'MR Média ({MR_mean:.2f})')
    UCL_MR = MR_mean * 3.267
    plt.axhline(UCL_MR, color='red', linestyle='--', label=f'UCL_MR ({UCL_MR:.2f})')

    plt.title(f"Carta de Faixa Móvel — MR (Modelo {model_name})")
    plt.xlabel("Observação")
    plt.ylabel("MR (|Δ Consumo|)")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.savefig(f"{save_dir}/carta_MR_modelo_{model_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n✔ Figuras salvas em: {save_dir}")


# ================================
#  RODAR POR MODELO
# ================================
models = df['Modelo'].unique()
print("Modelos encontrados:", models)

for m in models:
    sub_df = df[df['Modelo'] == m]
    print(f"\n==============================")
    print(f"PROCESSANDO MODELO {m}")
    print(f"Linhas: {len(sub_df)}")
    plot_control_chart(sub_df, m)
